# Running the whole network

`01_one_check.ipynb` drives one check by hand. Here the loop just runs — and
because results transfer upstream, watching the whole network is watching the
dependency ladder work: **terminal reaches build first; everything else waits
for its downstream neighbour.**

With `build_model` and `run_nd_scenarios` both in place, the wave actually
travels. A terminal builds its model, runs its normal-depth library, and the
reach above it can then build — because that build needs the downstream
max-q stage transfer line, and the reach above that waits in turn. Each reach
climbs two rungs, and the network drains from its outlets upward.

It stops where the data does. A terminal that names no lake or coast has no
outflow polygon to drain through and no job can invent one, so it reports
`awaiting_inputs` and everything above it stays `waiting_downstream`. In the test
network the two clip-edge reaches are exactly that case — the standstill is
the ladder holding, not a failure.

Two modes, one switch:

| `RUN_FOREVER` | behaviour |
|---|---|
| `False` | stop at steady state: nothing in flight and a pass that submitted nothing |
| `True` | keep going the way a service would; interrupt the cell to stop |

Interrupting is always safe — every fact the loop needs is in the database
before the step that wrote it returns.

**Prerequisites:** stack up, the network seeded (`scripts/seed.py`), and both
job images available.

**On runtime.** A normal-depth library is one hydraulic simulation per
discharge, and the discharges are chosen adaptively, so a single reach can take
minutes to hours — these passes are far slower than model-only ones. CPU and
GPU are two separate builds of the run job rather than one image with a switch,
so `USE_GPU` below picks which is submitted. The CPU build is the default and
is slower still.

In [ ]:
import time

import pandas as pd

from recon import check, db, jobs, processing, queue
from recon.config import settings
from recon.workers import LocalDockerRunner, job_env

# CPU or GPU is a choice of IMAGE now, not a flag inside one: `_job_key` maps
# (solver, hardware) to a published build. Set this True on a machine with the
# NVIDIA Container Toolkit; it selects the GPU build and passes the device.
USE_GPU = False


def build_runner():
    # job_env() carries the S3 configuration both clients inside the images
    # need — boto3 reads AWS_ENDPOINT_URL, GDAL ignores it and wants its own —
    # so this is one call rather than a list of variables to get right by hand.
    return LocalDockerRunner(
        network=settings.docker_network,
        env_vars=job_env(),
        platform=settings.docker_platform,
        gpus="all" if USE_GPU else None,
        volumes=[f"{settings.docker_data_dir}:/data:ro"] if settings.docker_data_dir else [])


def tally():
    return db.one("""
        SELECT count(*) FILTER (WHERE state = 'finished')           AS finished,
               count(*) FILTER (WHERE state = 'waiting_downstream') AS waiting,
               count(*) FILTER (WHERE state = 'awaiting_inputs')    AS awaiting,
               count(*) FILTER (WHERE state = 'in_flight')          AS in_flight,
               count(*) FILTER (WHERE state = 'resting')            AS resting,
               count(*) FILTER (WHERE state = 'halted')             AS halted,
               count(*)                                             AS total
        FROM reach_status""")


runner = build_runner()
nd_job = check.RUN_ND_JOBS[("lisflood", USE_GPU)]
print(f"images  {runner.images['build_model']}")
print(f"        {runner.images[nd_job]}")
print(f"network {runner.network}")
print(tally())

## Settings

The cap is on **how many jobs run at once**, and on nothing else. Every due
reach is checked every pass; `run_check(..., may_submit=False)` observes,
records proofs and notes who it waits on, but starts no work.

That separation is load-bearing rather than tidy. Observation is the only way
the loop ever learns a job finished, so suppressing checks to hold the cap
means a completed library can sit in storage unnoticed while the loop waits on
an unrelated job — and the reach then looks unbuilt and gets submitted again.
Checking is cheap and cannot start work by itself; only submission needs a
limit.

How many jobs may run at once is still an open question in the design doc, so
until it is settled the number belongs to whoever runs the loop. Match it to the
machine: a normal-depth scenario used ~3.4 cores here, so two saturate 8.

In [ ]:
RUN_FOREVER = False
MAX_IN_FLIGHT = 4
INTERVAL = 8

## The loop

One pass is two questions put to the database, in order:

1. **which jobs are we waiting on?** — poll them; clear markers for the
   finished; request checks. First, so freed capacity is usable this pass.
2. **which reaches are due?** — check each: observe, gap, act.

Neither question needs anything remembered from the previous pass.

In [ ]:
started = time.time()
passes = 0
try:
    while True:
        passes += 1
        for outcome in jobs.status_pass(runner):
            if outcome["status"] in ("succeeded", "failed"):
                print(f"    reach {outcome['reach_id']} {outcome['step']}: "
                      f"{outcome['status']} - {outcome['action']}")

        submitted = 0
        for row in queue.due_reaches():
            # Every due reach is checked every pass; the cap limits SUBMISSION
            # only. Skipping checks to hold the cap would stop the loop noticing
            # finished work, because observation is the only way it ever does.
            at_cap = len(processing.in_flight()) >= MAX_IN_FLIGHT
            if check.run_check(row["reach_id"], runner,
                               may_submit=not at_cap, gpu=USE_GPU).submitted_ref:
                submitted += 1

        now = tally()
        print(f"[{time.time() - started:5.0f}s] pass {passes:3d}  "
              f"finished {now['finished']:>3}/{now['total']}  waiting {now['waiting']:>3}  "
              f"awaiting {now['awaiting']:>2}  in flight {now['in_flight']:>2}  "
              f"resting {now['resting']:>2}  halted {now['halted']:>2}  submitted {submitted}")

        # Steady means nothing is running and nothing was started — not that
        # every reach is finished. A network with reaches awaiting inputs
        # settles below 100%, and that is the correct place for it to stop.
        steady = now["in_flight"] == 0 and submitted == 0 and passes > 1
        if steady and not RUN_FOREVER:
            print(f"\nsteady after {passes} passes in {time.time() - started:.0f}s")
            break
        time.sleep(INTERVAL)
except KeyboardInterrupt:
    print("\nstopped by hand; nothing lost — in-flight jobs are recorded in the database")

## Where the network stands

`finished`, `waiting_downstream` and `awaiting_inputs` together should cover the
network. A reach is `finished` only when **both** its claims are current — a
model alone does not count — so this table is the ladder's progress, not just
the build's.

In [ ]:
display(pd.DataFrame(db.query(
    "SELECT state, count(*) AS reaches FROM reach_status GROUP BY state ORDER BY reaches DESC")))

display(pd.DataFrame(db.query("""
    SELECT rn.is_terminal,
           count(*)                                              AS reaches,
           count(*) FILTER (WHERE rs.model_id IS NOT NULL)        AS with_model,
           count(*) FILTER (WHERE rs.nd_materialized)             AS with_nd_library,
           sum(rs.nd_discharges)                                  AS scenarios
    FROM reach_status rs JOIN reach_network rn USING (reach_id)
    GROUP BY 1 ORDER BY 1 DESC""")))

## The wait graph

Every waiting reach points at the neighbour it waits for — recorded so a
viewer can draw this without recomputing any gap. Chains drain upstream one
rung at a time; a chain whose root is `awaiting_inputs` will not drain at all until
that reach's water body is authored.

In [ ]:
pd.DataFrame(db.query("""
    SELECT p.blocked_on_reach_id                    AS waits_on,
           ds.state                                 AS its_state,
           count(*)                                 AS reaches_waiting
    FROM reach_status rs
    JOIN reach_processing p USING (reach_id)
    JOIN reach_status ds ON ds.reach_id = p.blocked_on_reach_id
    WHERE rs.state = 'waiting_downstream'
    GROUP BY 1, 2 ORDER BY reaches_waiting DESC"""))

## Anything parked

In [ ]:
halted = db.query("SELECT reach_id, consecutive_failures, halted_at, last_error "
                  "FROM reach_processing WHERE halted")
if halted:
    display(pd.DataFrame(halted))
    print("\nafter fixing the cause:  processing.clear_halt(reach_id)")
else:
    print("nothing halted")